In [8]:
import os

os.environ["JAVA_HOME"] = r"C:\Program Files\Microsoft\jdk-17.0.20.101-hotspot"

os.environ["HADOOP_HOME"] = r"C:\hadoop"

os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

print("JAVA_HOME =", os.environ["JAVA_HOME"])
print("HADOOP_HOME =", os.environ["HADOOP_HOME"])
print(
    "winutils exists =",
    os.path.exists(r"C:\hadoop\bin\winutils.exe"),
)
print(
    "hadoop.dll exists =",
    os.path.exists(r"C:\hadoop\bin\hadoop.dll"),
)

JAVA_HOME = C:\Program Files\Microsoft\jdk-17.0.20.101-hotspot
HADOOP_HOME = C:\hadoop
winutils exists = True
hadoop.dll exists = True


In [9]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [10]:
!set PYTHON_GIL=0

In [11]:
%load_ext autoreload
%autoreload 2

import os

# 1. Force single-threading ONLY for the import phase
os.environ["OMP_NUM_THREADS"] = "1"
from credit_risk import run_pipeline

# 2. Immediately restore multi-threading for your training process
# Set this to your actual CPU physical core count (e.g., "4", "8", "12")
os.environ["OMP_NUM_THREADS"] = "16"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Setup Path

In [12]:
from pathlib import Path
import os

if "project_path" not in globals():
    project_path = Path.cwd().parent
    os.chdir(project_path)

print("Project path:", project_path)

Project path: c:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk


### Run Main Pipeline

In [13]:
import os
import time
import threading
import psutil


def monitor_resources(interval=5):

    current_pid = os.getpid()

    while True:
        try:
            memory = psutil.virtual_memory()

            python_ram = psutil.Process(current_pid).memory_info().rss / (1024**3)

            java_processes = []

            for proc in psutil.process_iter(["pid", "name", "memory_info"]):
                try:
                    name = (proc.info["name"] or "").lower()

                    if "java" in name:
                        ram = proc.info["memory_info"].rss / (1024**3)

                        java_processes.append((proc.info["pid"], ram))

                except (
                    psutil.NoSuchProcess,
                    psutil.AccessDenied,
                ):
                    continue

            java_ram = sum(ram for _, ram in java_processes)

            print(
                f"[MONITOR] "
                f"System={memory.percent:.1f}% "
                f"Available={memory.available / (1024**3):.1f}GB | "
                f"Python={python_ram:.1f}GB | "
                f"Java={java_ram:.1f}GB | "
                f"CPU={psutil.cpu_percent(interval=1):.1f}%"
            )

            time.sleep(interval)

        except Exception as e:
            print(f"[MONITOR] {e}")

            break

In [14]:
run_pipeline(project_path)

11:25:33  INFO      utils.spark   Creating Spark session: master=local[*] driver_memory=15g shuffle_partitions=256
11:25:33  INFO      utils.spark   Spark session created: version=4.2.0 default_parallelism=256
11:25:33  INFO      pipeline      ━━ Pipeline started ━━ project=C:\Users\vorad\OneDrive\Desktop\Projects\mortgage-credit-risk
11:25:33  INFO      ingest        Ingestion skipped by configuration
11:25:33  INFO      data_preprocess  Preprocessing skipped by configuration
11:25:33  INFO      reporting     Data-quality reporting skipped by configuration
11:25:33  INFO      modelling     Vintages being considered in model training pipeline: 2018, 2019, 2020, 2021
11:25:33  INFO      modelling     Vintages being considered for OOT: 2022
11:25:33  INFO      modelling     Loading development vintages: 2018, 2019, 2020, 2021
11:25:33  INFO      modelling     Loading OOT vintages: 2022
11:25:33  INFO      modelling.data_spark  Spark reading modelling dataset: approach=behavioral vintage=

Py4JJavaError: An error occurred while calling o852.fit.
: org.apache.spark.SparkException: surrogate cannot be computed. All the values in delinquent_accrued_interest,months_since_last_paid_installment are Null, Nan or missingValue(NaN)
	at org.apache.spark.ml.feature.Imputer.fit(Imputer.scala:204)
	at org.apache.spark.ml.feature.Imputer.fit(Imputer.scala:116)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)
	at java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke(NativeMethodAccessorImpl.java:77)
	at java.base/jdk.internal.reflect.DelegatingMethodAccessorImpl.invoke(DelegatingMethodAccessorImpl.java:43)
	at java.base/java.lang.reflect.Method.invoke(Method.java:569)
	at py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:244)
	at py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
	at py4j.Gateway.invoke(Gateway.java:282)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.